#### Competitor Data Cleaning & Scoring Analysis

**Goal**  
Load the simulated competitor projects dataset (2023–2025),  
perform cleaning and validation,  
create key derived scores (handover reliability, first-time buyer appeal, sustainability strength, positioning index),  
and run initial aggregate analysis to identify patterns and prepare for gap visualization.

**Input:** `data/competitor_projects_2023_2025.csv`  
**Output:** `data/cleaned_competitor_projects_2023_2025.csv` (with scores)  


**Focus Reminder**  
This project is about competitive positioning.  
We are building tools to spot where competitors are weak so a new player can differentiate.

#### Imports & Setup

In [6]:
import pandas as pd
import numpy as np
import os

print("Libraries loaded. Ready for cleaning & scoring.")

Libraries loaded. Ready for cleaning & scoring.


#### Load Data & Initial Inspection

In [7]:
INPUT_PATH = 'data/competitor_projects_2023_2025.csv'

df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(df)} projects")
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isna().sum())

# Quick type fixes
df['launch_year'] = df['launch_year'].astype(int)
df['handover_year'] = df['handover_year'].astype(int)
df['sustainability_score'] = df['sustainability_score'].astype(int)
df['first_time_focus'] = df['first_time_focus'].astype(int)

Loaded 80 projects

Columns: ['project_id', 'developer', 'emirate', 'area', 'property_type', 'status', 'launch_year', 'handover_year', 'avg_pps_aed', 'sustainability_score', 'first_time_focus', 'total_units_planned', 'project_name']

Missing values:
 project_id              0
developer               0
emirate                 0
area                    0
property_type           0
status                  0
launch_year             0
handover_year           0
avg_pps_aed             0
sustainability_score    0
first_time_focus        0
total_units_planned     0
project_name            0
dtype: int64


In [8]:
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,project_id,developer,emirate,area,property_type,status,launch_year,handover_year,avg_pps_aed,sustainability_score,first_time_focus,total_units_planned,project_name
0,1,Emaar,Dubai,Palm Jumeirah,Apartment,Ready,2024,2024,2659.0,0,0,500,Emaar Bay 32
1,2,DAMAC,Dubai,Jumeirah Village Circle (JVC),Apartment,Ready,2025,2024,1331.0,0,0,500,DAMAC Pinnacle 12
2,3,Emaar,Dubai,Business Bay,Apartment,Off-plan,2024,2025,2027.0,0,0,500,Emaar Residences 12
3,4,Emaar,Dubai,Business Bay,Apartment,Off-plan,2024,2025,1928.0,0,0,500,Emaar Pinnacle 26
4,5,Omniyat,Dubai,Arjan,Villa,Off-plan,2025,2025,1407.0,0,0,500,Omniyat Gardens 58


#### Cleaning & Validation

In [9]:
# 1. Realistic bounds
df['avg_pps_aed'] = df['avg_pps_aed'].clip(700, 3500)
df['total_units_planned'] = df['total_units_planned'].clip(30, 2000)

# 2. Flag any clipped rows (for transparency)
df['clipped_price'] = (df['avg_pps_aed'].isin([700, 3500])).astype(int)
df['clipped_units'] = (df['total_units_planned'].isin([30, 2000])).astype(int)

print(f"Rows with price clipped: {df['clipped_price'].sum()}")
print(f"Rows with units clipped: {df['clipped_units'].sum()}")

# 3. Handover consistency check (should not be before launch)
invalid_handover = df[df['handover_year'] < df['launch_year']]
if not invalid_handover.empty:
    print(f"Warning: {len(invalid_handover)} rows with handover before launch – fixing to launch year.")
    df.loc[df['handover_year'] < df['launch_year'], 'handover_year'] = df['launch_year']

# 4. No missing critical columns (simulation should be clean, but check)
critical_cols = ['developer', 'emirate', 'area', 'property_type', 'status', 'avg_pps_aed']
print("\nMissing in critical columns:\n", df[critical_cols].isna().sum())

Rows with price clipped: 0
Rows with units clipped: 0

Missing in critical columns:
 developer        0
emirate          0
area             0
property_type    0
status           0
avg_pps_aed      0
dtype: int64


#### Derived Scores (Core for Positioning Analysis)

We create interpretable scores (0–10 scale where possible) to compare competitors.

In [10]:
# 1. Handover Reliability Score (higher = better / more reliable)
# Ready projects = max score; earlier handover in period = better
df['handover_reliability_score'] = 0.0

# Ready = 10
df.loc[df['status'] == 'Ready', 'handover_reliability_score'] = 10.0

# Off-plan: score based on how early in period (2023 launch + 2025 handover = best)
offplan_mask = df['status'] == 'Off-plan'
df.loc[offplan_mask, 'handover_reliability_score'] = np.where(
    df.loc[offplan_mask, 'launch_year'] == 2023,
    7.0 + (df.loc[offplan_mask, 'handover_year'] - 2023) * 1.5,
    5.0 + (df.loc[offplan_mask, 'handover_year'] - 2024) * 2.0
)
df['handover_reliability_score'] = df['handover_reliability_score'].clip(0, 10)

# 2. First-Time Buyer Appeal Score (0–10)
# Higher if: lower pps, higher sustainability, first_time_focus=1
df['pps_normalized'] = (df['avg_pps_aed'] - df['avg_pps_aed'].min()) / (df['avg_pps_aed'].max() - df['avg_pps_aed'].min())
df['pps_score'] = 10 * (1 - df['pps_normalized'])  # lower price = higher score

df['first_time_appeal_score'] = (
    0.4 * df['pps_score'] +
    0.3 * (df['sustainability_score'] / 3 * 10) +
    0.3 * (df['first_time_focus'] * 10)
).round(1)

# 3. Sustainability Strength (already 0–3, normalize to 0–10)
df['sustainability_strength'] = (df['sustainability_score'] / 3 * 10).round(1)

# 4. Overall Positioning Index (simple weighted score for differentiation potential)
df['positioning_index'] = (
    0.35 * df['first_time_appeal_score'] +
    0.30 * df['handover_reliability_score'] +
    0.25 * df['sustainability_strength'] +
    0.10 * (df['total_units_planned'] / df['total_units_planned'].max() * 10)  # scale bonus
).round(1)

print("\nNew scores created:")
print(df[['developer', 'project_name', 'avg_pps_aed', 'first_time_appeal_score',
         'handover_reliability_score', 'sustainability_strength', 'positioning_index']].head(8))


New scores created:
   developer         project_name  avg_pps_aed  first_time_appeal_score  \
0      Emaar         Emaar Bay 32       2659.0                      1.0   
1      DAMAC    DAMAC Pinnacle 12       1331.0                      3.6   
2      Emaar  Emaar Residences 12       2027.0                      2.2   
3      Emaar    Emaar Pinnacle 26       1928.0                      2.4   
4    Omniyat   Omniyat Gardens 58       1407.0                      3.4   
5      Emaar       Emaar Hills 90       2696.0                      3.9   
6  Ellington   Ellington Hills 28       1662.0                      4.9   
7  Binghatti   Binghatti Tower 12       2399.0                      3.5   

   handover_reliability_score  sustainability_strength  positioning_index  
0                        10.0                      0.0                4.0  
1                        10.0                      0.0                4.9  
2                         7.0                      0.0                3.5  

In [13]:
import os

# 1. Ensure the 'data' folder exists
os.makedirs('data', exist_ok=True)

# 2. Define save path
OUTPUT_CLEAN_PATH = 'data/cleaned_competitor_projects_2023_2025.csv'

# 3. Save the DataFrame (df should already contain all scores from Cell 5)
df.to_csv(OUTPUT_CLEAN_PATH, index=False)

# 4. Confirmation messages
print(f"\nCleaned & scored dataset successfully saved to:")
print(f"→ {OUTPUT_CLEAN_PATH}")

print(f"File size: {os.path.getsize(OUTPUT_CLEAN_PATH) / 1024:.1f} KB")
print("File exists now?", os.path.exists(OUTPUT_CLEAN_PATH))

print("\nQuick preview of saved file (first 3 rows):")
print(pd.read_csv(OUTPUT_CLEAN_PATH).head(3))


Cleaned & scored dataset successfully saved to:
→ data/cleaned_competitor_projects_2023_2025.csv
File size: 12.1 KB
File exists now? True

Quick preview of saved file (first 3 rows):
   project_id developer emirate                           area property_type  \
0           1     Emaar   Dubai                  Palm Jumeirah     Apartment   
1           2     DAMAC   Dubai  Jumeirah Village Circle (JVC)     Apartment   
2           3     Emaar   Dubai                   Business Bay     Apartment   

     status  launch_year  handover_year  avg_pps_aed  sustainability_score  \
0     Ready         2024           2024       2659.0                     0   
1     Ready         2025           2025       1331.0                     0   
2  Off-plan         2024           2025       2027.0                     0   

   ...  total_units_planned         project_name clipped_price  clipped_units  \
0  ...                  500         Emaar Bay 32             0              0   
1  ...              